# Notebook 7: Training Pipeline

## Learning Objectives
- Understand language modeling loss
- Implement training loop
- Learn about MTP (Multi-Token Prediction)
- Understand FP8 training concepts

## Task 1: Language Modeling Loss

### HINT: Cross-Entropy Loss
```python
# Next token prediction: predict token t+1 from token t
loss = cross_entropy(logits[:, :-1, :], labels[:, 1:])
```

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Simple dataset for demonstration
def create_dummy_data(vocab_size, num_samples, seq_len):
    """Create random training data"""
    input_ids = torch.randint(0, vocab_size, (num_samples, seq_len))
    # Labels are shifted by 1
    labels = torch.randint(0, vocab_size, (num_samples, seq_len))
    return TensorDataset(input_ids, labels)

vocab_size = 50000
seq_len = 32
dataset = create_dummy_data(vocab_size, 1000, seq_len)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)

print(f"Dataset size: {len(dataset)}")
print(f"Batches: {len(dataloader)}")
print(f"Batch shape: {next(iter(dataloader))[0].shape}")

## Task 2: Training Loop

### HINT: Standard PyTorch Training
```python
for epoch in range(num_epochs):
    for batch in dataloader:
       
        logits = optimizer.zero_grad() model(input_ids)
        loss = compute_loss(logits, labels)
        loss.backward()
        optimizer.step()
```

In [ ]:
# Use model from previous notebook
from notebook_06_block import DeepSeekV3

config = {
    'vocab_size': 50000,
    'hidden_size': 256,  # Smaller for faster training
    'num_layers': 2,
    'num_heads': 4,
    'num_kv_heads': 1,
    'num_experts': 4,
    'num_active_experts': 2,
    'intermediate_size': 512
}

model = DeepSeekV3(config)
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)

# Training loop
num_epochs = 2

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    
    for batch_idx, (input_ids, labels) in enumerate(dataloader):
        # Forward pass
        logits = model(input_ids)
        
        # Compute loss (shift for next-token prediction)
        # logits: [batch, seq, vocab] → [batch*seq, vocab]
        # labels: [batch, seq] → [batch*seq]
        loss_fn = nn.CrossEntropyLoss()
        
        # Shift for next token prediction
        logits_shifted = logits[:, :-1, :].contiguous().view(-1, vocab_size)
        labels_shifted = labels[:, 1:].contiguous().view(-1)
        
        loss = loss_fn(logits_shifted, labels_shifted)
        
        # Backward
        optimizer.zero_grad()
        loss.backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        
        optimizer.step()
        
        total_loss += loss.item()
        
        if batch_idx % 50 == 0:
            print(f"Epoch {epoch}, Batch {batch_idx}, Loss: {loss.item():.4f}")
    
    avg_loss = total_loss / len(dataloader)
    print(f"Epoch {epoch} complete. Avg loss: {avg_loss:.4f}")

## Task 3: Multi-Token Prediction (MTP)

### HINT: Predict Multiple Tokens
```
Standard: predict only t+1
MTP: predict t+1, t+2, t+3 simultaneously
```

This improves sample efficiency and helps with long-range dependencies!

In [ ]:
class MTPLoss(nn.Module):
    """Multi-Token Prediction Loss"""
    def __init__(self, num_predictions=2):
        super().__init__()
        self.num_predictions = num_predictions
        
    def forward(self, hidden_states, labels, model):
        """
        hidden_states: [batch, seq, hidden]
        labels: [batch, seq]
        """
        total_loss = 0
        
        for k in range(1, self.num_predictions + 1):
            # Predict token at position t+k
            # Shift hidden states and labels
            if k < hidden_states.shape[1]:
                hs_k = hidden_states[:, :-k, :]
                labels_k = labels[:, k:]
                
                # Get logits
                logits_k = model.lm_head(hs_k)
                
                # Loss
                loss_k = nn.CrossEntropyLoss()(logits_k.view(-1, logits_k.shape[-1]), labels_k.view(-1))
                total_loss += loss_k
        
        return total_loss / self.num_predictions

# MTP concept demonstration
print("=== Multi-Token Prediction ===")
print("Instead of predicting just token t+1:")
print("  MTP predicts: t+1, t+2, t+3, ...")
print("  Each prediction has its own head")
print("  Loss = average of all predictions")

## Task 4: FP8 Training Concepts

### HINT: Mixed Precision
```python
# Standard: FP32 (full precision)
# FP8: 8-bit floating point (much faster, less memory)

# DeepSeek V3 uses FP8 training:
# - Most ops in FP8
# - Master weights in FP32
# - Dynamic loss scaling
```

In [ ]:
# Demonstrate mixed precision training (PyTorch)
from torch.cuda.amp import autocast, GradScaler

# Check if CUDA available
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

print("\n=== FP8/Mixed Precision Training ===")
print("Benefits:")
print("  - 2-4x faster training")
print("  - 50% less memory")
print("  - Similar accuracy to FP32")

# PyTorch AMP example (automatic mixed precision)
print("\nPyTorch AMP usage:")
print("""
scaler = GradScaler()

with autocast():
    logits = model(input_ids)
    loss = compute_loss(logits, labels)

scaler.scale(loss).backward()
scaler.step(optimizer)
scaler.update()
""")

## Verification

Complete these checks:
1. ✅ Can implement language modeling loss
2. ✅ Can write complete training loop
3. ✅ Understand MTP concept
4. ✅ Know FP8/mixed precision benefits